In [ ]:
using Bessels, ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz

ω_b = ω_b_0 #+ Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ=1/10; # 1/ns from Leon's paper
ϕ_quantum=1/2

ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d # Pump Frequency

ϵ_0 = 0.25
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2

κ_2 = 4 * g2^2/κ;

In [ ]:
function two_photon_CL_3rd_order(t_max;
        circuit_params=(ω_a=4*2*pi, ω_b=7.0*2*pi, E_J = 37*2*pi, E_Ca=0.0121, E_Cb=0.0705, ϵ_0 = 0.6*pi, κ_a=0, κ_b=1/10, ϵ_d=0.24, ω_d=ω_b),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞, cat_phase=0.0)

    
    ω_a,ω_b, E_J, E_Ca, E_Cb, ϵ_0, κ_a, κ_b, ϵ_d,ω_d = circuit_params
    C_a_inv=2*E_Ca # hbar/e^2 GHz
    C_b_inv=2*E_Cb # hbar/e^2 GHz
    Z_a=C_a_inv/ω_a # Impedance of the resonator a
    Z_b=C_b_inv/ω_b # Impedance of the resonator b

    ϕ_quantum = 1/2 # ħ/e 

    function drive(t)
        return ϵ_d * cos(ω_d * t+ cat_phase) #We consider a drive resonant with the mode b
    end

    ω_p=2ω_a-ω_d
    function epsilon(t)
        return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
    end

    # Definition of the second order ODE
    function damped_two_photon_2(ddu, du, u, p, t)
        ϕ_a = u[1]
        ϕ_b = u[2]
        
        ddu[1] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(t))*(-(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2 / 2) / ϕ_quantum  - 2 * κ_a * du[1]
        ddu[2] = -ω_b^2*ϕ_b - ω_b *drive(t)*sqrt(2) - 2 * κ_b * du[2] + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(t))*(-(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2 / 2) / ϕ_quantum 
    end

    # Initial conditions
    ϕ_a_0, q_a_0 = init_mode_a

    ϕ_b_0, q_b_0 = init_mode_b
    
    ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b))
    ϕ_dot_0 = vcat(q_a_0 * sqrt(Z_a)*ω_a,q_b_0 * sqrt(Z_b)*ω_b)

    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (50 * ω_b) # Time step for the simulation

    prob = SecondOrderODEProblem(damped_two_photon_2,ϕ_dot_0, ϕ_0, tspan)
    sol = solve(prob, VelocityVerlet(), dt=dt)

    # Extract solution arrays
    ϕ_dot_CL_a = [u[1] for u in sol.u]*sqrt(Z_a)
    ϕ_CL_a = [u[3] for u in sol.u]*sqrt(Z_a)
    q_CL_a = ϕ_dot_CL_a / C_a_inv

    ϕ_dot_CL_b = [u[2] for u in sol.u]*sqrt(Z_b)
    ϕ_CL_b = [u[4] for u in sol.u]*sqrt(Z_b)
    q_CL_b = ϕ_dot_CL_b / C_b_inv

    return ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, sol.t
end

function two_photon_1d_chain_3rd_order(n, t_max; chain_params=nothing, bath=nothing, bath_params=nothing,
     circuit_params=(ω_a=4*2*pi,ω_b=7.05*2*pi,E_J = 37*2*pi, E_Ca=0.0121, E_Cb=0.0705, ϵ_0 = 0.6*pi,κ=1/10,ϵ_d=0.24,ω_d=7.0*2*pi),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,cat_phase=0.0,minimal_a=false,emission=false, fast_no_chain=true)



    # Intialization of the circuit parameters
    ω_a, ω_b, E_J, E_Ca, E_Cb, ϵ_0, κ, ϵ_d, ω_d = circuit_params
    C_a_inv=2*E_Ca # hbar/e^2 GHz
    C_b_inv=2*E_Cb # hbar/e^2 GHz
    Z_a=C_a_inv/ω_a # Impedance of the resonator a
    Z_b=C_b_inv/ω_b # Impedance of the resonator b
    

    ϕ_quantum=1/2 #flux quantum in hbar/e


    # Initialization of the bath parameters
    if chain_params===nothing
        AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
        onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
    else
        onsite_energies, hopping_energies, system_chain_coupling = chain_params
    end

    function drive(t)
        return ϵ_d * cos(ω_d * t + cat_phase) #We consider a drive resonant with the mode b
    end

    ω_p=2ω_a-ω_d
    function epsilon(t)
        return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
    end


    temp_buf = zeros(n)
    function two_photon_ODE(ddu,du,u,p,t)
        ϕ_a = u[1]
        ϕ_b = u[2]
        Φ=u[3:end]

        
        ddu[1] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(t))*((ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2)/2 / ϕ_quantum
        ddu[2] = -ω_b^2*ϕ_b + ω_b * system_chain_coupling * sqrt(Z_b) * Φ[1] - ω_b *drive(t)*sqrt(2) + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(t))*((ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2)/2 / ϕ_quantum
        mul!(temp_buf, Mϕ_chain, Φ)
        @. temp_buf = -temp_buf
        temp_buf[1] += system_chain_coupling * sqrt(Z_b) * ϕ_b
        mul!(view(ddu, 3:length(ddu)), Mq_chain, temp_buf)
        return nothing
    end


    gq=vcat(- hopping_energies)
    gϕ=vcat(hopping_energies)

    Mq_chain=SymTridiagonal(onsite_energies, gq)
    Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)

    Mq_full=SymTridiagonal(vcat([ω_a,ω_b], onsite_energies), vcat([0,0],gq))

    #Initial conditions
    ϕ_a_0,q_a_0 = init_mode_a

    ϕ_b_0,q_b_0 = init_mode_b


    ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b), fill(0.0, n))
    q_0 = vcat(q_a_0 * sqrt(Z_a),q_b_0 * sqrt(Z_b), fill(0.0, n))
    ϕ_dot_0 = Mq_full*q_0


    #Run the simulation
    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (5 * maximum(vcat(ω_p,ω_a,ω_b,onsite_energies))) # Time step for the simulation
    prob = SecondOrderODEProblem(two_photon_ODE, ϕ_dot_0, ϕ_0, tspan)

    if fast_no_chain
        save_idxs=[1,2,n+3,n+4] # We only save the system DoFs (ϕ_a, ϕ_b) and not the chain DoFs (Φ)
        sol = solve(prob, VelocityVerlet(),dt=dt,
        saveat=dt,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
        )

        ϕ_dot = [getindex.(sol.u,i) for i in 1:2]
        ϕ = [getindex.(sol.u,i+2) for i in 1:2]
    
        ϕ_a = ϕ[1] * sqrt(Z_a)
        q_a = ϕ_dot[1]/ω_a / sqrt(Z_a)
        if minimal_a
            return(ϕ_a,q_a,sol.t)
        end

        ϕ_b = ϕ[2] * sqrt(Z_b)
        q_b = ϕ_dot[2]/ω_b / sqrt(Z_b)

        t_list = sol.t
        return(ϕ_a,ϕ_b,q_a,q_b,t_list)
    end

    save_idxs=vcat(3:n+2,n+5:n+4+n) # We save the chain DoFs (Φ)
    saveat= t_max # We save every 100 steps
    n_red = length(save_idxs) # Number of reduced DoFs to save
    sol = solve(prob, VelocityVerlet(),dt=dt,
    saveat=saveat,                      # how often to save (or choose a coarser step)
    save_idxs=save_idxs,            # only store system DoFs
    save_everystep=false            # do not store intermediate solver steps
    )
    ϕ_dot = [getindex.(sol.u,i)[end] for i in 1:n]
    ϕ = [getindex.(sol.u,n+i)[end] for i in 1:n]
    return(ϕ,ϕ_dot,saveat)
end

In [ ]:
t_max=2/κ
α_fin=2.2
cat_phase=pi
q_a_0 = -α_fin* sqrt(2/Z_a) *cos(cat_phase/2) # Initial condition for mode a
ϕ_a_0 = -α_fin* sqrt(2*Z_a) *sin(cat_phase/2)
ϕ_a, ϕ_CL_b, q_a, q_CL_b, t_list = two_photon_CL_3rd_order(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=0,κ_b=κ,ϵ_d=ϵ_d,ω_d=ω_b_0+Lamb_shift*2*pi),
        init_mode_a=(ϕ_a_0=ϕ_a_0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞, cat_phase=cat_phase);

In [ ]:
Re_a = ϕ_a / sqrt(2*Z_a)
Im_a = q_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_a * cos(ω_a * t_list) + q_a*Z_a* sin(-ω_a * t_list)
q_rotating_a =@. -ϕ_a/Z_a * sin(-ω_a * t_list) + q_a* cos(ω_a * t_list)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

p1=plot(Re_a,Im_a)
p2=plot(Re_rotating_a,Im_rotating_a)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
n=7000

bath=Exp_decay
bath_params = bath_params=(ω_0=50*2*pi, ω_cut_off=30*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
t_max=4/κ
ϕ_a,ϕ_b,q_a,q_b,t_list = two_photon_1d_chain_3rd_order(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = 0,κ=κ, ϵ_d=0, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=1.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=false)

In [ ]:
plot(t_list*κ,ϕ_b)

In [ ]:
α_fin=sqrt(ϵ_d/(2*g2))
cat_phase=0
q_a_0 = -α_fin* sqrt(2/Z_a) *cos(cat_phase/2) # Initial condition for mode a
ϕ_a_0 = -α_fin* sqrt(2*Z_a) *sin(cat_phase/2)
ϕ_a,q_a,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=ϕ_a_0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,cat_phase=cat_phase,minimal_a=true,emission=false)

In [ ]:
Re_a = ϕ_a / sqrt(2*Z_a)
Im_a = q_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_a * cos(ω_a * t_list) + q_a*Z_a* sin(-ω_a * t_list)
q_rotating_a =@. -ϕ_a/Z_a * sin(-ω_a * t_list) + q_a* cos(ω_a * t_list)

Re_rotating_a = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a = q_rotating_a * sqrt(Z_a/2)

p1=plot(Re_a,Im_a)
p2=plot(Re_rotating_a,Im_rotating_a)

layout = (1,2)
plot(p1,p2,layout=layout, xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=["Resonator a" "Resonator a RF"], legend=false,
size=(layout[2], layout[1]).*size_px_car,
margin = 10Plots.mm)

In [ ]:
gq=vcat(- chain_params[2])
gϕ=vcat(chain_params[2])

Mq_chain=SymTridiagonal(chain_params[1], gq)
Mϕ_chain=SymTridiagonal(chain_params[1], gϕ)

A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)

P_chain_inv = inv(P_chain);  # Inverse of P_chain, used for computing S_chain

Dq_chain = diag(P_chain'*(Mq_chain \ P_chain));  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain);

In [ ]:
cat_phase_list=-2pi:pi/5:2pi
t_max=15/κ
Re_rotating_a_list = Vector{Vector{Float64}}(undef, length(cat_phase_list))
Im_rotating_a_list = Vector{Vector{Float64}}(undef, length(cat_phase_list))
t_list_list = Vector{Vector{Float64}}(undef, length(cat_phase_list))
for (index, cat_phase_i) in enumerate(cat_phase_list)
     q_a_0 = 0.01*cos(cat_phase_i/2)* sqrt(2/Z_a) # Initial condition for mode a
     ϕ_a_0 = 0.01*sin(cat_phase_i/2)* sqrt(2*Z_a)
     ϕ_a, _, q_a, _, t_list = two_photon_CL_3rd_order(t_max;
          circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=0,κ_b=κ, ϵ_d=ϵ_d, ω_d=ω_b_0+Lamb_shift*2*pi),
          init_mode_a=(ϕ_a_0=ϕ_a_0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,cat_phase=cat_phase_i)
     Re_a = ϕ_a / sqrt(2*Z_a)
     Im_a = q_a * sqrt(Z_a/2)

     Re_rotating_a =@. Re_a * cos(ω_a * t_list) + Im_a* sin(-ω_a * t_list)
     Im_rotating_a =@. -Re_a * sin(-ω_a * t_list) + Im_a* cos(ω_a * t_list)

     Re_rotating_a_list[index] = Re_rotating_a
     Im_rotating_a_list[index] = Im_rotating_a
     t_list_list[index] = t_list
end

In [ ]:
θ= 0:0.01:2pi # Circle for the cat state
# Animate by updating the point
@gif for index in 1:length(cat_phase_list)
    alpha_round = round(cat_phase_list[index]/2, digits = 1)
    plot(
    Re_rotating_a_list[index][1:10:end], Im_rotating_a_list[index][1:10:end],
    xlims=(-3,3),
    ylim=(-3,3),
    label="cat phase = $alpha_round",
    legend=:topright,
    marker=:circle,
    markersize=2,
    xlabel=L"\Re(a)",
    ylabel=L"\Im(a)",
    title="Evolution towards the cat state",
    size=size_px_car,
    )
    plot!(sqrt(ϵ_d/(2*g2))*cos.(θ), sqrt(ϵ_d/(2*g2))*sin.(θ), label="", color=:red, linewidth=2, linestyle=:dash)
end every 1  # you can adjust the frame rate

In [ ]:
cat_phase_list=-2pi:pi/20:2pi
α_fin=sqrt(ϵ_d/(2*g2))
t_max=15/κ
Phase = Array{Float64}(undef, length(cat_phase_list), n)
Amplitude = Array{Float64}(undef, length(cat_phase_list), n)
Energy = Array{Float64}(undef, length(cat_phase_list), n)
for (index, cat_phase_i) in enumerate(cat_phase_list)
     q_a_0 = 0.01* sqrt(2/Z_a) *cos(cat_phase_i/2) # Initial condition for mode a
     ϕ_a_0 = 0.01* sqrt(2*Z_a) *sin(cat_phase_i/2)
     ϕ_chain,ϕ_dot_chain,t_list= two_photon_1d_chain_3rd_order(n, t_max; chain_params=chain_params,
          circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
          init_mode_a=(ϕ_a_0=ϕ_a_0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,cat_phase=cat_phase_i,fast_no_chain=false)



     S = P_chain_inv *  ϕ_chain
     S_dot = P_chain_inv * ϕ_dot_chain
     Q_S = 2 * Dq_chain .* S_dot
     Phase[index, :] = angle.(S+1im*Q_S)
     Amplitude[index, :] = abs.(S+1im*Q_S)
     Energy[index,:]=  Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;
end

In [ ]:
Phase_centered = [mod.(Phase[i,:].+pi .- Phase[1,:], 2pi) for i in 1:size(Phase, 1)];

In [ ]:
# Animate by updating the point
f_a=ω_a/2pi
f_b=ω_b/2pi
f_p=abs(ω_p/2pi)
@gif for index in 1:length(cat_phase_list)
    alpha_round = round(cat_phase_list[index]/2, digits = 1)
    plot(
    ωk_chain[23:end]/2pi, Energy[index,23:end],
    ylim=(1e-10, 1e2),
    xlim=(0,16),
    yaxis=:log10,
    label="cat phase = $(alpha_round)",
    legend=:topright,
    xlabel=L"\omega_k/2\pi"*" (GHz)",
    ylabel="Energy",
    title="Energy of the chain modes",
    )
end every 1  # you can adjust the frame rate

In [ ]:
ϕ,ϕ_dot,t_list = two_photon_1d_chain_3rd_order(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0 = 0.01* sqrt(2*Z_a) *sin(pi/3),q_a_0 =  0.01* sqrt(2/Z_a) *cos(pi/3)), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞, fast_no_chain=false)

# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0 = 0.01* sqrt(2*Z_a) *sin(pi/3),q_a_0 =  0.01* sqrt(2/Z_a) *cos(pi/3)), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain_cos = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
plot(ωk_chain[23:end]/2pi, Energies_normal_modes_chain_cos[23:end], label = "Cosine")
plot!(ωk_chain[23:end]/2pi, Energies_normal_modes_chain[23:end], alpha=0.7, label="3rd order")
plot!(ylim=(1e-10, 1e2),
    xlim=(0,16),
    yaxis=:log10,
    legend=:topright,
    xlabel=L"\omega_k/2\pi"*" (GHz)",
    ylabel="Energy",
    title="Energy of the chain modes",
    )
#savefig("../../Figures/Two-photon/Comparison_emission_spectrum_3rd_order_cosine.pdf")

In [ ]:
Phase_diff=diff(Phase_centered)
Phase_diff_bis = [getindex.(Phase_diff, i) for i in 1:length(Phase_diff[1])]
Phase_diff_bis_binary = [map(x -> x > 5 ? 1 : 0, subvec) for subvec in Phase_diff_bis]

Winding_number = [
    sum(d) for d in Phase_diff_bis_binary
]
Residual = pi .- Phase_centered[end, :][1];

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain,yaxis=:log, ylims=(1e-10, 1e2),ylabel="Energy stored in the chain",xlabel="Frequency (GHz)")

plot!(twinx(),ωk_chain[1:end-1]/2pi,Winding_number[1:end-1].+Residual[1:end-1], label="", ylabel="Non-integer winding number", color=Perso_palette_hex[2], linewidth=2, grid=true)
plot!(title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, xlims=(0.1, 16))
savefig("../../Figures/Two-photon/Winding_number_chain_3rd_order.pdf")